## Boston Bautista

Question: Which industries show the most consistent profitability and which experience the highest volatility?

In [1]:
from pyspark.sql import SparkSession
from math import sqrt

def csv_split(line):
    return line.split(",")

In [2]:
# Create Spark session connected to localhost cluster
spark = (
    SparkSession.builder.appName("Stock Market Analysis")
    # .master("spark://localhost:7077")
    .getOrCreate()
)

# Get Spark context from session
sc = spark.sparkContext

# Set log level to reduce verbosity
sc.setLogLevel("WARN")

print("✅ Connected to Spark cluster!")
print(f"Spark Version: {spark.version}")
print(f"Master: {sc.master}")
print(f"App ID: {sc.applicationId}")

25/11/22 13:34:18 WARN Utils: Your hostname, MacBook-Pro-246.local resolves to a loopback address: 127.0.0.1; using 10.0.0.101 instead (on interface en0)
25/11/22 13:34:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/22 13:34:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Connected to Spark cluster!
Spark Version: 3.5.4
Master: local[*]
App ID: local-1763847260927


In [3]:
num_csv_path = "./data/processed/merged/num_2020.csv"
pre_csv_path = "./data/processed/merged/pre_2020.csv"
sub_csv_path = "./data/processed/merged/sub_2020.csv"
tag_csv_path = "./data/processed/merged/tag_2020.csv"


num_rdd = sc.textFile(num_csv_path)
pre_rdd = sc.textFile(pre_csv_path)
sub_rdd = sc.textFile(sub_csv_path)
tag_rdd = sc.textFile(tag_csv_path)

# print size of each RDD
print(f"Num RDD size: {num_rdd.count()}")
print(f"Pre RDD size: {pre_rdd.count()}")
print(f"Sub RDD size: {sub_rdd.count()}")
print(f"Tag RDD size: {tag_rdd.count()}")

Num RDD size: 11493263


Pre RDD size: 2746310
Sub RDD size: 24940
Tag RDD size: 298803


In [4]:
num_head = num_rdd.first().split(",")
sub_head = sub_rdd.first().split(",")

n_adsh  = num_head.index("adsh")
n_tag   = num_head.index("tag")
n_uom   = num_head.index("uom")
n_seg   = num_head.index("segments")
n_coreg = num_head.index("coreg")
n_val   = num_head.index("value")
s_adsh  = sub_head.index("adsh")
s_sic   = sub_head.index("sic")

In [5]:
num_body_rdd = (num_rdd.filter(lambda line: line != num_rdd.first()).map(csv_split).filter(lambda row: len(row) == len(num_head)))

sub_body_rdd = (sub_rdd.filter(lambda line: line != sub_rdd.first()).map(csv_split).filter(lambda row: len(row) == len(sub_head)))

In [6]:
tags = {"NetIncomeLoss", "Assets"}

num_filtered_rdd = (num_body_rdd.filter(lambda row: row[n_tag] in tags and row[n_uom] == "USD" and row[n_seg] == "" and row[n_coreg] == ""))

In [8]:
def pairs(row):
    adsh = row[n_adsh]
    tag  = row[n_tag]
    val  = (row[n_val])
    if val is None:
        return None

    if tag == "NetIncomeLoss":
        return (adsh, (val, None))
    
    else:
        return (adsh, (None, val))

num_pairs_rdd = num_filtered_rdd.map(pairs).filter(lambda x: x is not None)

In [9]:
sc.stop()